# Ep. 02 — LLM, memória e limites de contexto

Quatro passos práticos: chamada simples, esquecimento, histórico e estouro da janela.


## 1. Chamada simples

Fazemos uma chamada a um LLM e mostramos a resposta.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

def ask(messages):
    """messages: string OU lista de dicts {role, content}."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Defina OPENAI_API_KEY e rode de novo.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

resposta = ask("Lembre deste segredo: a palavra-código é ORBIT-77.")
print(resposta)


## 2. Sem histórico

Agora pergunte ao modelo o que ele acabou de falar.


In [ ]:
# Nova chamada — sem passar o histórico da célula anterior
resposta2 = ask("O que você acabou de dizer?")
print(resposta2)


## 3. Montar o histórico

Para o modelo "lembrar", basta uma lista de mensagens `role` / `content`, com a última no final.


In [ ]:
historico = [
    {"role": "user", "content": "Lembre deste segredo: a palavra-código é ORBIT-77."},
    {"role": "assistant", "content": resposta},
    {"role": "user", "content": "Qual é a palavra-código?"},
]

resposta3 = ask(historico)
print(resposta3)


## 4. Estouro da janela

Se o histórico crescer demais, a janela de contexto estoura e o provedor devolve erro.


In [ ]:
# Cresce o histórico até a API falhar (ex.: 400 context length exceeded)
historico_grande = [
    {"role": "user", "content": "Lembre deste segredo: a palavra-código é ORBIT-77."},
    {"role": "assistant", "content": resposta},
]

bloco = "x" * 50_000  # texto grande só para encher o contexto

try:
    while True:
        historico_grande.append({"role": "user", "content": bloco})
        historico_grande.append({"role": "assistant", "content": "ok"})
        print(f"mensagens no histórico: {len(historico_grande)}")
        ask(historico_grande)
except Exception as e:
    print("Erro do provedor:")
    print(e)
